# ULTRON on Google Colab

Run the **full** Ultron app (dashboard, brain, tools, memory, SSE) on Colab's hardware instead of your own machine, reachable from your browser through a tunnel.

**What runs where**
- The FastAPI server, brain, tools, and dashboard run on the Colab VM.
- `run_shell` / `read_file` / `system_stats` act on the **Colab VM** (a throwaway Google container), not your computer.
- The LLM "brain" is still a cloud API (Groq here) — the same as running locally.

**Heads up: Colab sessions are temporary.** When the runtime resets, `ultron.db` and anything you didn't push to GitHub is gone. Cell 6 (optional) mounts Google Drive to persist the memory DB across sessions.

Run the cells top to bottom.

## 1. Clone (or update) the repo

In [41]:
import os
if not os.path.isdir("/content/ultron"):
    !git clone https://github.com/Razoradams9/ultron.git /content/ultron
%cd /content/ultron
!git pull
!ls


/content/ultron
Already up to date.
 colab.ipynb	    run.py				     ultron.db
 DESIGN.md	    server.log				     voice
 README.md	    ultron
 requirements.txt  'Ultron_ Best Lines & Moments.mp3.mpeg'


## 2. Install dependencies

The core app is lightweight. `pyngrok` is for the tunnel. Voice deps (Chatterbox/torch) are **not** installed here — do that in the optional voice cell only if you want spoken replies.

In [30]:
!pip install -q -r requirements.txt
!pip install -q pyngrok nest_asyncio

## 3. Set your Groq API key

Paste your key from [console.groq.com/keys](https://console.groq.com/keys). It lives only in this session's memory — it is not written to the repo.

Optional: uncomment to switch persona to the polite butler.

In [31]:

import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your GROQ_API_KEY: ").strip()
os.environ["ULTRON_PROVIDER"] = "groq"
# os.environ["ULTRON_PERSONA"] = "jarvis"   # sardonic 'ultron' is the default

print("Key set." if os.environ["GROQ_API_KEY"] else "No key entered — the brain won't respond.")

Paste your GROQ_API_KEY: ··········
Key set.


In [32]:
import urllib.request
try:
    r = urllib.request.urlopen("http://127.0.0.1:8000/api/status", timeout=8)
    print("STATUS OK:", r.read().decode())
except Exception as e:
    print("ERROR:", e)


ERROR: <urlopen error [Errno 111] Connection refused>


In [33]:
%cd /content/ultron
import threading, time, uvicorn, nest_asyncio, traceback
nest_asyncio.apply()

def _serve():
    try:
        uvicorn.run("ultron.server:app", host="0.0.0.0", port=8000, log_level="info")
    except Exception:
        traceback.print_exc()

threading.Thread(target=_serve, daemon=True).start()
time.sleep(5)
print("server thread started — try the dashboard link again, then check output here")


/content/ultron


INFO:     Started server process [1542]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


server thread started — try the dashboard link again, then check output here


## 4. (Optional) Get an ngrok token

A free [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) authtoken gives you a stable tunnel to the dashboard. Skip this cell to use Colab's built-in port proxy instead (Cell 5 handles both).

In [34]:
from getpass import getpass

token = getpass("Paste your ngrok authtoken (or press Enter to skip): ").strip()
if token:
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    print("ngrok token set.")
else:
    print("Skipping ngrok — will use Colab's port proxy.")

Paste your ngrok authtoken (or press Enter to skip): ··········
Skipping ngrok — will use Colab's port proxy.


## 5. Launch the server + open the dashboard

Starts uvicorn in the background on port 8000, then exposes it. Click the printed URL to open the ULTRON control room.

Re-run this cell if you edit code and want a fresh server.

In [43]:
import threading, time, uvicorn, nest_asyncio

nest_asyncio.apply()  # Colab already runs an event loop; let uvicorn share it

PORT = 8000

def _serve():
    uvicorn.run("ultron.server:app", host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=_serve, daemon=True).start()
time.sleep(4)  # let it bind

public_url = None
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(PORT).public_url
    print("ULTRON dashboard:", public_url)
except Exception as e:
    print("ngrok unavailable (", e, ") — falling back to Colab proxy below.")
    try:
        from google.colab.output import eval_js
        print("ULTRON dashboard:", eval_js(f"google.colab.kernel.proxyPort({PORT})"))
    except Exception as e2:
        print("Colab proxy also unavailable:", e2)

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
ERROR:pyngrok.process.ngrok:t=2026-09-24T19:33:53+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-09-24T19:33:53+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-09-24T19:33:53+0000 lvl=c

ngrok unavailable ( The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n. ) — falling back to Colab proxy below.
ULTRON dashboard: https://8000-gpu-t4-s-kkb-ass1c0-484okk2q7mfj-c.asia-southeast1-0.prod.colab.dev


## 6. (Optional) Persist memory across sessions

By default `ultron.db` is created in the repo folder and vanishes when the runtime resets. Run this cell **before Cell 5** to keep the conversation log + facts on your Google Drive instead.

In [36]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.makedirs('/content/drive/MyDrive/ultron', exist_ok=True)
# # config.py reads ultron.db from the project root; symlink it to Drive:
# db = '/content/drive/MyDrive/ultron/ultron.db'
# link = '/content/ultron/ultron.db'
# if not os.path.islink(link):
#     if os.path.exists(link):
#         os.remove(link)
#     os.symlink(db, link)
# print('Memory persisted at', db)

## 7. (Optional) GPU-accelerated cloned voice

The voice service is the only heavy piece. Colab gives you a free GPU, so it runs far faster here than on CPU.

**First:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU*, then re-run cells 1–5.

This cell installs the voice deps, starts `voice/server.py` on port 8001 (it auto-detects CUDA via the new `VOICE_DEVICE` logic), and points the main app at it by setting `VOICE_URL`. Re-run Cell 5 afterward so the server picks up `VOICE_URL`.

In [42]:
import os, glob, threading, time

# install Chatterbox without wrecking Colab's CUDA torch
!pip install -q --no-deps chatterbox-tts
!pip install -q pykakasi==2.3.0 pyloudnorm resemble-perth s3tokenizer spacy-pkuseg conformer librosa soundfile

import torch
assert torch.cuda.is_available(), "Not on GPU — Runtime > Change runtime type > T4 GPU, then rerun."
os.environ["VOICE_DEVICE"] = "cuda"

# register your uploaded clip as the voiceprint
cands = [f for f in glob.glob("/content/*") if f.lower().endswith(('.mp3','.wav','.m4a','.mpeg','.flac','.ogg'))]
cands = [c for c in cands if not c.endswith(("ref.wav","cloned.wav","ultron_test.wav"))]
cands.sort(key=os.path.getmtime, reverse=True)
assert cands, "No clip in /content — upload one via the folder icon first."
import librosa, soundfile as sf
os.makedirs("/content/ultron/voice/samples", exist_ok=True)
y, sr = librosa.load(cands[0], sr=24000, mono=True)
sf.write("/content/ultron/voice/samples/reference.wav", y, 24000, subtype="PCM_16")
print("voiceprint from:", cands[0], f"({len(y)/sr:.0f}s)")

# start the voice service on GPU
%cd /content/ultron
def _serve_voice():
    import uvicorn
    uvicorn.run("voice.server:app", host="127.0.0.1", port=8001, log_level="info")
threading.Thread(target=_serve_voice, daemon=True).start()
time.sleep(5)
os.environ["VOICE_URL"] = "http://127.0.0.1:8001"
print("voice service starting on :8001 — now RE-RUN the app launch cell (Cell 5).")


voiceprint from: /content/Ultron (1).mp3 (59s)
/content/ultron


INFO:     Started server process [1542]
INFO:     Waiting for application startup.


[voice] preload complete — service warm


INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8001): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


voice service starting on :8001 — now RE-RUN the app launch cell (Cell 5).


In [37]:
import urllib.request, os
print("VOICE_URL env:", os.environ.get("VOICE_URL"))
try:
    r = urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=10)
    print("VOICE HEALTH:", r.read().decode())
except Exception as e:
    print("VOICE SERVICE UNREACHABLE:", e)


VOICE_URL env: None
VOICE SERVICE UNREACHABLE: <urlopen error [Errno 111] Connection refused>
